# Lemari flat-lay: deteksi banyak garmen sekali foto

Tiga hal yang dibangun di sini, semuanya dari dua dataset publik:

1. **Komposisi** — garmen dipotong dari foto produk dengan matte U²-Net yang sudah dipakai aplikasi, lalu ditempel 2–6 buah ke satu kanvas. Kotak tiap garmen otomatis tepat, jadi tidak ada anotasi manual.
2. **Detektor satu kelas** — `yolov8n` hanya belajar "di sini ada garmen", tanpa lima kategori. Kotak ganda yang muncul pada model sekarang hilang karena tidak ada kategori yang bisa bertengkar.
3. **Classifier** — dilatih pada crop yang diambil dari komposisi yang sama, supaya masukannya sama dengan yang nanti diberi detektor.

Di sel terakhir kamu bisa mengunggah foto sendiri dan melihat kotak serta nama tiap garmen.

**Sebelum Run All:** panel kanan → Accelerator **GPU T4**, Internet **On**. Perkiraan: matte ±25 menit, detektor ±30 menit, classifier ±15 menit.

Catatan data: notebook ini memakai `clothing-dataset-small` (CC0, 10 kelas, publik) supaya bisa dijalankan tanpa mengunggah apa pun. `Clothes_Dataset` 15 kelas ada di laptop, jadi untuk versi itu nanti ia perlu diunggah sebagai dataset Kaggle privat dan `SOURCE` diganti.


In [ ]:
# Langkah 1/9 — cek internet, pasang paket, cek runtime.
import importlib.util
import socket
import subprocess
import sys


def online(host="pypi.org"):
    try:
        socket.gethostbyname(host)
        return True
    except OSError:
        return False


if not online():
    raise SystemExit(
        "Sesi ini tidak punya internet: nama pypi.org tidak bisa diterjemahkan.\n"
        "  1. Panel kanan -> Settings -> Internet -> On.\n"
        "     Kalau yang muncul 'Verify phone number', verifikasi dulu di kaggle.com/settings.\n"
        "  2. Run -> Restart & Clear Cell Outputs (mengubah Internet hanya berlaku untuk sesi baru).\n"
        "  3. Run All lagi."
    )
print("internet      ok")


def ensure(package):
    if importlib.util.find_spec(package) is not None:
        print(f"ada           {package}")
        return
    print(f"memasang      {package}", flush=True)
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], capture_output=True, text=True)
    if done.returncode != 0:
        tail = (done.stderr or done.stdout or "").strip().splitlines()[-4:]
        raise SystemExit(f"gagal memasang {package} (kode {done.returncode}).\n" + "\n".join(tail))
    if importlib.util.find_spec(package) is None:
        raise SystemExit(f"{package} terpasang tetapi tidak terlihat oleh kernel; restart session lalu ulangi")


for package in ("ultralytics", "ai-edge-litert", "huggingface_hub"):
    ensure(package)

import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")
print(f"cuda          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu           {torch.cuda.get_device_name(0)}")


## 2. Ambil repo

Skrip komposisinya ada di repo (`ml_workspace/scripts/compose_flatlay.py`), bukan di notebook. Ganti
`REPO_REF` hanya kalau skripnya ada di branch lain.


In [ ]:
# Langkah 2/9 — clone repo.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "feat/flatlay-pipeline"
REPO_DIR = Path("/kaggle/temp/gayaku")


def run(command, **kwargs):
    print("+", " ".join(str(part) for part in command))
    completed = subprocess.run(command, text=True, **kwargs)
    if completed.returncode != 0:
        raise SystemExit(f"perintah gagal ({completed.returncode}): {' '.join(map(str, command))}")
    return completed


if not (REPO_DIR / ".git").is_dir():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
COMPOSER = REPO_DIR / "ml_workspace/scripts/compose_flatlay.py"
if not COMPOSER.is_file():
    raise SystemExit(f"{COMPOSER} tidak ada di {REPO_REF}")
print("repo siap", REPO_DIR)


## 3. Unduh sumber garmen dan contoh foto uji

`clothing-dataset-small` (CC0) berisi 3.068 foto latih, satu garmen per foto, 10 kelas.
`Fashion-1K` (OpenRAIL) berisi 1.000 foto flat-lay: **dipakai sebagai contoh uji saja**, bukan
sebagai latar komposisi. Fotonya berisi garmen, dan latar bergarmen tanpa kotak justru mengajari
detektor untuk mengabaikan garmen.


In [ ]:
# Langkah 3/9 — unduh sumber garmen dan contoh uji.
from huggingface_hub import snapshot_download

CC0 = Path("/kaggle/temp/clothing-dataset-small")
if not (CC0 / "train").is_dir():
    run(["git", "clone", "--depth", "1",
         "https://github.com/alexeygrigorev/clothing-dataset-small.git", str(CC0)])

SAMPLES = Path("/kaggle/temp/fashion1k/images")
if not list(SAMPLES.glob("*.jpg")):
    snapshot_download(
        repo_id="Humanbased-AI/Fashion-1K",
        repo_type="dataset",
        local_dir="/kaggle/temp/fashion1k",
        allow_patterns=["images/*.jpg"],
    )

classes = sorted(p.name for p in (CC0 / "train").iterdir() if p.is_dir())
print("kelas     ", len(classes), classes)
print("contoh uji", len(list(SAMPLES.glob('*.jpg'))), "foto")


## 4. Bangun komposisi

Tiap foto sumber dihitung matte-nya sekali, hasilnya di-cache; setelah itu komposisi hanya menempel.
Latar dibuat prosedural (ubin, papan, karpet, dinding) supaya tidak ada garmen tanpa kotak yang ikut
tertempel. `--max-sources 300` membatasi biaya matte; naikkan kalau ingin variasi lebih banyak.


In [ ]:
# Langkah 4/9 — komposisi train + val.
import time

FLATLAY = Path("/kaggle/temp/flatlay")
started = time.time()
run(
    [sys.executable, str(COMPOSER),
     "--source", str(CC0 / "train"),
     "--out", str(FLATLAY),
     "--count", "2000",
     "--val", "0.12",
     "--min-items", "2",
     "--max-items", "6",
     "--max-sources", "300",
     "--size", "800"],
    cwd=str(REPO_DIR),
)
print(f"selesai dalam {(time.time() - started) / 60:.1f} menit")

scenes = sorted((FLATLAY / "images/train").glob("*.jpg"))
crops = sorted((FLATLAY / "crops/train").rglob("*.jpg"))
print("scene latih", len(scenes), "| crop latih", len(crops))


## 5. Periksa satu komposisi

Kalau kotaknya meleset di sini, latihan detektor juga akan mewarisi kesalahan itu — lebih murah
melihatnya sekarang.


In [ ]:
# Langkah 5/9 — gambar satu scene beserta kotaknya.
from PIL import Image, ImageDraw
from IPython.display import display

sample = sorted((FLATLAY / "images/train").glob("*.jpg"))[0]
label_file = FLATLAY / "labels/train" / (sample.stem + ".txt")
canvas = Image.open(sample).convert("RGB")
draw = ImageDraw.Draw(canvas)
for line in label_file.read_text().splitlines():
    _, cx, cy, w, h = (float(v) for v in line.split())
    x0, y0 = (cx - w / 2) * canvas.width, (cy - h / 2) * canvas.height
    x1, y1 = (cx + w / 2) * canvas.width, (cy + h / 2) * canvas.height
    draw.rectangle([x0, y0, x1, y1], outline=(255, 60, 60), width=4)
display(canvas.resize((520, 520)))
print(sample.name, "|", len(label_file.read_text().splitlines()), "kotak")


## 6. Latih detektor satu kelas

`yolov8n`, 640 px, satu kelas `garment`. Yang dilaporkan adalah mAP50 pada scene validasi — scene
yang tidak dipakai latih, tetapi tetap sintetis. Angka itu menunjukkan detektornya belajar, bukan
bahwa ia siap dipakai pada foto asli.


In [ ]:
# Langkah 6/9 — latih detektor.
from ultralytics import YOLO

detector = YOLO("yolov8n.pt")
detector.train(
    data=str(FLATLAY / "data.yaml"),
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/temp/runs",
    name="flatlay-det",
    exist_ok=True,
    seed=42,
    patience=10,
)
print("bobot terbaik", detector.trainer.save_dir)


## 7. Latih classifier pada crop komposisi

Crop diambil dari scene jadi, termasuk lantai dan potongan garmen tetangga di tepinya — sama seperti
masukan yang nanti diberikan detektor. Ini yang membuat angka val-nya masuk akal.


In [ ]:
# Langkah 7/9 — latih classifier.
classifier = YOLO("yolov8n-cls.pt")
classifier.train(
    data=str(FLATLAY / "crops"),
    epochs=25,
    imgsz=224,
    batch=64,
    device=0,
    project="/kaggle/temp/runs",
    name="flatlay-cls",
    exist_ok=True,
    seed=42,
    patience=8,
)
print("bobot terbaik", classifier.trainer.save_dir)


## 8. Uji: unggah satu foto sendiri

Unggah foto baju diletakkan (boleh beberapa sekaligus dalam satu foto). Notebook menjalankan
detektor, memotong tiap kotak, menamainya dengan classifier, lalu menampilkan gambar berlabel dan
tabel item. Kalau tidak ada yang diunggah, ia memakai satu scene validasi sebagai contoh.


In [ ]:
# Langkah 8/9 — deteksi + klasifikasi pada foto yang kamu unggah.
from ultralytics import YOLO as _YOLO
from PIL import Image, ImageDraw
from IPython.display import display
import io

BEST_DET = sorted(Path("/kaggle/temp/runs").glob("flatlay-det*/weights/best.pt"))[-1]
BEST_CLS = sorted(Path("/kaggle/temp/runs").glob("flatlay-cls*/weights/best.pt"))[-1]
print("detektor   ", BEST_DET)
print("classifier ", BEST_CLS)

det = _YOLO(str(BEST_DET))
cls = _YOLO(str(BEST_CLS))

try:
    from google.colab import files  # noqa: F401  (Kaggle tidak punya; pakai widget di bawah)
except Exception:
    pass

photo = None
try:
    from ipywidgets import FileUpload
    upload = FileUpload(accept="image/*", multiple=False)
    display(upload)          # klik "Upload", pilih foto, lalu jalankan ulang sel ini
    if upload.value:
        name = next(iter(upload.value))
        photo = Image.open(io.BytesIO(upload.value[name]["content"])).convert("RGB")
        print("diunggah", name, photo.size)
except Exception as error:
    print("widget unggah tidak tersedia:", error)

if photo is None:
    samples = sorted(SAMPLES.glob("*.jpg")) if SAMPLES.is_dir() else []
    sample = samples[0] if samples else sorted((FLATLAY / "images/val").glob("*.jpg"))[0]
    photo = Image.open(sample).convert("RGB")
    print("contoh foto uji:", sample.name, "(bukan foto sintetis)" if samples else "(scene sintetis)")

boxes = det.predict(photo, imgsz=640, conf=0.25, verbose=False)[0]
rows = []
drawn = photo.copy()
painter = ImageDraw.Draw(drawn)
for index, box in enumerate(boxes.boxes):
    x0, y0, x1, y1 = (float(v) for v in box.xyxy[0])
    pad_x, pad_y = (x1 - x0) * 0.08, (y1 - y0) * 0.08
    crop = photo.crop((max(0, x0 - pad_x), max(0, y0 - pad_y),
                       min(photo.width, x1 + pad_x), min(photo.height, y1 + pad_y)))
    guess = cls.predict(crop, imgsz=224, verbose=False)[0]
    name = guess.names[int(guess.probs.top1)]
    score = float(guess.probs.top1conf)
    rows.append((index + 1, name, f"{score:.2f}", f"{float(box.conf[0]):.2f}"))
    painter.rectangle([x0, y0, x1, y1], outline=(255, 60, 60), width=5)
    painter.text((x0 + 6, max(0, y0 - 18)), f"{name} {score:.2f}", fill=(255, 255, 255))

display(drawn.resize((min(900, drawn.width), int(drawn.height * min(900, drawn.width) / drawn.width))))
print(f"{len(rows)} garmen terdeteksi")
for row in rows:
    print(f"  {row[0]}. {row[1]:16s} classifier {row[2]}  detektor {row[3]}")


## 9. Bungkus hasil

Zip berisi bobot, laporan, dan manifest komposisi. Kirim angkanya ke thread sebelum ada keputusan
mengganti aset di aplikasi.


In [ ]:
# Langkah 9/9 — zip artefak.
import zipfile

ZIP = Path("/kaggle/working/gayaku_flatlay.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for pattern in ("flatlay-det*", "flatlay-cls*"):
        for run_dir in Path("/kaggle/temp/runs").glob(pattern):
            for path in sorted(run_dir.rglob("*")):
                if path.is_file() and path.suffix in {".pt", ".csv", ".png", ".yaml", ".json"}:
                    archive.write(path, str(path.relative_to("/kaggle/temp/runs")))
    for path in (FLATLAY / "flatlay_manifest.json", FLATLAY / "data.yaml"):
        if path.is_file():
            archive.write(path, path.name)
print(ZIP, f"{ZIP.stat().st_size / 1024 / 1024:.1f} MB")
